# 01 — Data preparation and seasonal discovery

**Research question.** Do individual pitchers exhibit different movement profiles during seasons with Colorado versus adjacent seasons elsewhere?

This notebook retains the project's foundational Baseball Savant exports, MLB Rockies roster acquisition, transition construction, team-share checks, and exploratory movement comparisons. It uses the original computational methods, with the project directory centralized in the setup cell. **Seasonal observations do not isolate the altitude effect:** Rockies seasons also include road appearances.

**Inputs:** the sixteen `Full *.csv` exports from Baseball Savant (excluding the redundant `Full Slidersa.csv` file). **Outputs:** the `rockies_pitchers_2015_2026.csv`, `coors_master_pitch_types_long.csv`, `coors_pitch_transitions.csv`, `rockies_innings_share.csv` and the changeup robustness results used downstream.

Run this first. It may make MLB StatsAPI requests. Original cell outputs are preserved for review but have **not** been rerun or verified in a clean session here.


In [ ]:
# Shared project configuration — change this path if you use a different Drive folder.
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Coors Project'
DATA_PATH = Path(PROJECT_DIR)
if not DATA_PATH.is_dir():
    raise FileNotFoundError(f'Create or locate your project folder: {DATA_PATH}')
print('Project data:', DATA_PATH)


### Step 1 — Download the Rockies pitching records


In [1]:
# ============================================================
# COORS FIELD PITCHING ANALYSIS
# STEP 1: IMPORT BASEBALL SAVANT DATA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Connect Google Drive
# Drive mounted by notebook setup

# Set project directory
DATA_PATH = Path(PROJECT_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Folder not found: {DATA_PATH}")

# Locate the new expanded datasets
csv_files = sorted(DATA_PATH.glob("Full *.csv"))

# Exclude the duplicate slider dataset
csv_files = [
    file for file in csv_files
    if file.name != "Full Slidersa.csv"
]

# Load datasets separately
datasets = {}
summary = []

for file in csv_files:

    df = pd.read_csv(file, low_memory=False)

    datasets[file.stem] = df

    unique_seasons = (
        df[["player_id", "year"]]
        .drop_duplicates()
        .shape[0]
    )

    summary.append({
        "Dataset": file.stem,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Unique Pitcher-Seasons": unique_seasons
    })

# Display summary
data_summary = pd.DataFrame(summary)

print("BASEBALL SAVANT DATA IMPORT")
print("=" * 50)

print(f"Datasets loaded: {len(datasets)}")

display(data_summary)

# Preview general pitching data
print("\nFULL BASIC DATA PREVIEW")

display(datasets["Full Basic"].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASEBALL SAVANT DATA IMPORT
Datasets loaded: 16


,Dataset,Rows,Columns,Unique Pitcher-Seasons
0,Full 4 Seams,9357,13,9352
1,Full Basic,9357,8,9352
2,Full BreakingBalls General,9357,13,9352
3,Full ChangeUps,9357,12,9352
4,Full Curveballs,9357,12,9352
5,Full Cutters,9357,13,9352
6,Full Fastballs General,9357,13,9352
7,Full Forkballs,9357,13,9352
8,Full Knuckleballs,9357,13,9352
9,Full OffSpeed General,9357,13,9352



FULL BASIC DATA PREVIEW


,"last_name, first_name",player_id,year,pa,k_percent,bb_percent,barrel_batted_rate,hard_hit_percent
0,"Colon, Bartolo",112526,2015,815,16.7,2.9,6.1,34.7
1,"Hawkins, LaTroy",115629,2015,162,21.0,4.3,5.7,38.8
2,"Wolf, Randy",150116,2015,161,17.4,9.3,5.8,41.0
3,"Marquis, Jason",150302,2015,216,17.1,6.5,3.0,35.4
4,"Burnett, A.J.",150359,2015,699,20.5,7.0,5.6,38.7


### Step 2 — Rockies Pitching History


In [2]:
# ============================================================
# STEP 2: COLLECT COLORADO ROCKIES PITCHING HISTORY
# ============================================================

import requests
import time
import pandas as pd

BASE_URL = "https://statsapi.mlb.com/api/v1/stats"

ROCKIES_ID = 115
YEARS = range(2015, 2027)

all_pitchers = []
failed_years = []

for year in YEARS:

    params = {
        "stats": "season",
        "group": "pitching",
        "teamId": ROCKIES_ID,
        "season": year,
        "sportIds": 1,
        "playerPool": "ALL",
        "gameType": "R"
    }

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=45
        )

        response.raise_for_status()
        data = response.json()

        season_pitchers = []

        for block in data.get("stats", []):
            for split in block.get("splits", []):

                player = split.get("player", {})
                team = split.get("team", {})
                stats = split.get("stat", {})

                if team.get("id") != ROCKIES_ID:
                    continue

                games = int(stats.get("gamesPlayed") or 0)

                if games == 0:
                    continue

                season_pitchers.append({
                    "player_id": player.get("id"),
                    "player_name": player.get("fullName"),
                    "year": year,
                    "team": "COL",
                    "games": games,
                    "games_started": stats.get("gamesStarted"),
                    "innings_pitched": stats.get("inningsPitched"),
                    "era": stats.get("era"),
                    "whip": stats.get("whip"),
                    "strikeouts": stats.get("strikeOuts"),
                    "walks": stats.get("baseOnBalls"),
                    "home_runs": stats.get("homeRuns")
                })

        if not season_pitchers:
            raise ValueError("No pitching records returned")

        all_pitchers.extend(season_pitchers)

        print(f"{year}: {len(season_pitchers)} pitchers")

    except Exception as e:
        failed_years.append(year)
        print(f"{year}: ERROR - {e}")

    time.sleep(0.2)

# Do not treat incomplete downloads as a complete roster
if failed_years:
    raise RuntimeError(
        f"Download incomplete. Failed seasons: {failed_years}"
    )

# Create DataFrame
rockies_pitchers = pd.DataFrame(all_pitchers)

# Standardize identifiers
rockies_pitchers["player_id"] = (
    rockies_pitchers["player_id"].astype(int)
)

rockies_pitchers["year"] = (
    rockies_pitchers["year"].astype(int)
)

# Verify uniqueness
assert not rockies_pitchers.duplicated(
    ["player_id", "year"]
).any(), "Duplicate pitcher-season records found"

# Sort
rockies_pitchers = rockies_pitchers.sort_values(
    ["year", "player_name"]
).reset_index(drop=True)

# Save to Google Drive
rockies_pitchers.to_csv(
    DATA_PATH / "rockies_pitchers_2015_2026.csv",
    index=False
)

# Summary
print("\n" + "=" * 50)
print("ROCKIES PITCHING DATA SUMMARY")
print("=" * 50)

print("Total pitcher-seasons:", len(rockies_pitchers))

print(
    "Unique Rockies pitchers:",
    rockies_pitchers["player_id"].nunique()
)

print("\nPitchers per season:")

display(
    rockies_pitchers.groupby("year")["player_id"]
    .nunique()
    .reset_index(name="pitchers")
)

print("\nDataset preview:")
display(rockies_pitchers.head(10))

2015: 30 pitchers
2016: 25 pitchers
2017: 21 pitchers
2018: 21 pitchers
2019: 31 pitchers
2020: 24 pitchers
2021: 26 pitchers
2022: 25 pitchers
2023: 34 pitchers
2024: 33 pitchers
2025: 31 pitchers
2026: 34 pitchers

ROCKIES PITCHING DATA SUMMARY
Total pitcher-seasons: 335
Unique Rockies pitchers: 165

Pitchers per season:


,year,pitchers
0,2015,30
1,2016,25
2,2017,21
3,2018,21
4,2019,31
5,2020,24
6,2021,26
7,2022,25
8,2023,34
9,2024,33



Dataset preview:


,player_id,player_name,year,team,games,games_started,innings_pitched,era,whip,strikeouts,walks,home_runs
0,444836,Aaron Laffey,2015,COL,3,0,7.1,3.68,1.50,3,3,1
1,493603,Adam Ottavino,2015,COL,10,0,10.1,0.00,0.48,13,2,0
2,457429,Boone Logan,2015,COL,60,0,35.1,4.33,1.61,44,17,3
3,453156,Brooks Brown,2015,COL,36,0,33.0,4.91,1.45,20,16,2
4,518452,Chad Bettis,2015,COL,20,20,115.0,4.23,1.41,98,42,11
5,543734,Chris Rusin,2015,COL,24,22,131.2,5.33,1.60,86,41,19
6,548357,Christian Bergman,2015,COL,30,4,68.1,4.74,1.42,37,15,8
7,543184,Christian Friedrich,2015,COL,68,0,58.1,5.25,1.71,45,25,5
8,571735,David Hale,2015,COL,17,12,78.1,6.09,1.47,61,20,14
9,572750,Eddie Butler,2015,COL,16,16,79.1,5.90,1.82,44,42,13


### Step 3. Create Master Data Set


In [3]:
# ============================================================
# STEP 3: BUILD THE MASTER PITCHING DATASET
# ============================================================

import pandas as pd
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)

# Pitch type, filename, Savant prefix, general category?
pitch_files = {
    "four_seam": ("Full 4 Seams.csv", "ff", False),
    "sinker": ("Full Sinkers.csv", "si", False),
    "cutter": ("Full Cutters.csv", "fc", False),
    "slider": ("Full Sliders.csv", "sl", False),
    "sweeper": ("Full Sweepers.csv", "st", False),
    "curveball": ("Full Curveballs.csv", "cu", False),
    "changeup": ("Full ChangeUps.csv", "ch", False),
    "splitter": ("Full Splitters.csv", "fs", False),
    "forkball": ("Full Forkballs.csv", "fo", False),
    "screwball": ("Full Screwballs.csv", "sc", False),
    "slurve": ("Full Slurve.csv", "sv", False),
    "knuckleball": ("Full Knuckleballs.csv", "kn", False),

    "fastball_group": (
        "Full Fastballs General.csv", "fastball", True
    ),
    "breaking_group": (
        "Full BreakingBalls General.csv", "breaking", True
    ),
    "offspeed_group": (
        "Full OffSpeed General.csv", "offspeed", True
    )
}

keys = ["player_id", "year"]

# ------------------------------------------------------------
# 1. Load general pitching statistics
# ------------------------------------------------------------

basic = pd.read_csv(DATA_PATH / "Full Basic.csv")

basic = basic.rename(
    columns={"last_name, first_name": "player_name"}
)

# Remove exact duplicate rows
basic = basic.drop_duplicates()

# Ensure uniqueness of player-season records
assert not basic.duplicated(keys).any(), (
    "Conflicting pitcher-season records in Full Basic"
)

# ------------------------------------------------------------
# 2. Combine individual pitch measurements
# ------------------------------------------------------------

pitch_datasets = []

for pitch_type, (filename, prefix, is_group) in pitch_files.items():

    df = pd.read_csv(DATA_PATH / filename)

    columns = {
        "last_name, first_name": "player_name",
        f"n_{prefix}_formatted": "usage_percent",
        f"{prefix}_avg_speed": "avg_speed",
        f"{prefix}_avg_spin": "avg_spin",
        f"{prefix}_avg_break_x": "avg_break_x",
        f"{prefix}_avg_break_z": "avg_break_z",
        f"{prefix}_avg_break_z_induced": "avg_break_z_induced",
        f"{prefix}_avg_break": "avg_break",
        f"{prefix}_range_speed": "range_speed"
    }

    # Check required columns exist
    missing = set(columns) - set(df.columns)

    if missing:
        raise ValueError(
            f"{filename} missing columns: {missing}"
        )

    # Keep relevant columns and standardize names
    df = df[
        keys + list(columns.keys())
    ].rename(columns=columns)

    # Remove exact duplicate records only
    df = df.drop_duplicates()

    # Remove rows without a recorded pitch velocity
    df = df.dropna(subset=["avg_speed"])

    df["pitch_type"] = pitch_type
    df["is_group"] = is_group

    pitch_datasets.append(df)

    print(f"{pitch_type}: {len(df):,} records")

# Combine all pitch types
all_pitches = pd.concat(
    pitch_datasets,
    ignore_index=True
)

# Standardize identifiers
for df in [all_pitches, basic]:
    df["player_id"] = pd.to_numeric(
        df["player_id"], errors="raise"
    ).astype("int64")

    df["year"] = pd.to_numeric(
        df["year"], errors="raise"
    ).astype("int64")

# Flag multiple pitch profiles for the same player-season
profile_keys = ["player_id", "year", "pitch_type"]

all_pitches["profile_variant"] = (
    all_pitches.groupby(profile_keys).cumcount() + 1
)

all_pitches["multiple_profiles"] = (
    all_pitches.groupby(profile_keys)["player_id"]
    .transform("size")
    .gt(1)
)

# ------------------------------------------------------------
# 3. Add general pitching statistics
# ------------------------------------------------------------

general_columns = [
    "player_id",
    "year",
    "pa",
    "k_percent",
    "bb_percent",
    "barrel_batted_rate",
    "hard_hit_percent"
]

master = all_pitches.merge(
    basic[general_columns],
    on=keys,
    how="left",
    validate="many_to_one"
)

assert len(master) == len(all_pitches), (
    "General statistics merge unexpectedly changed row count"
)

# ------------------------------------------------------------
# 4. Add Rockies history
# ------------------------------------------------------------

rockies = pd.read_csv(
    DATA_PATH / "rockies_pitchers_2015_2026.csv"
)

rockies["player_id"] = rockies["player_id"].astype(int)
rockies["year"] = rockies["year"].astype(int)

rockies_lookup = rockies[
    ["player_id", "year", "games", "innings_pitched"]
].rename(columns={
    "games": "rockies_games",
    "innings_pitched": "rockies_innings"
})

master = master.merge(
    rockies_lookup,
    on=keys,
    how="left",
    validate="many_to_one",
    indicator=True
)

master["rockies"] = master["_merge"].map({
    "both": "Y",
    "left_only": "N",
    "right_only": "N"
}).astype(str)

master = master.drop(columns="_merge")

# ------------------------------------------------------------
# 5. Save master dataset
# ------------------------------------------------------------

output_file = DATA_PATH / "coors_master_pitch_types_long.csv"

master.to_csv(output_file, index=False)

# ------------------------------------------------------------
# 6. Validate results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("MASTER DATASET SUMMARY")
print("=" * 55)

print("Total rows:", f"{len(master):,}")
print("Total columns:", len(master.columns))
print("Unique pitchers:", master["player_id"].nunique())

print(
    "Unique pitcher-seasons:",
    master[keys].drop_duplicates().shape[0]
)

rockies_records = master[master["rockies"] == "Y"]

print(
    "Rockies pitcher-seasons:",
    rockies_records[keys].drop_duplicates().shape[0]
)

print(
    "Unique Rockies pitchers:",
    rockies_records["player_id"].nunique()
)

print(
    "Rows with multiple pitch profiles:",
    master["multiple_profiles"].sum()
)

print("\nRockies pitcher-seasons by year:")

display(
    rockies_records[keys]
    .drop_duplicates()
    .groupby("year")
    .size()
    .reset_index(name="rockies_pitcher_seasons")
)

print("\nMaster dataset preview:")
display(master.head(10))

print("\nSaved to:", output_file)

four_seam: 8,802 records
sinker: 6,619 records
cutter: 3,110 records
slider: 6,804 records
sweeper: 1,775 records
curveball: 5,454 records
changeup: 6,965 records
splitter: 1,029 records
forkball: 17 records
screwball: 9 records
slurve: 144 records
knuckleball: 41 records
fastball_group: 9,264 records
breaking_group: 9,227 records
offspeed_group: 7,824 records

MASTER DATASET SUMMARY
Total rows: 67,084
Total columns: 23
Unique pitchers: 2543
Unique pitcher-seasons: 9352
Rockies pitcher-seasons: 322
Unique Rockies pitchers: 156
Rows with multiple pitch profiles: 58

Rockies pitcher-seasons by year:


,year,rockies_pitcher_seasons
0,2015,30
1,2016,25
2,2017,21
3,2018,21
4,2019,31
5,2020,23
6,2021,25
7,2022,24
8,2023,31
9,2024,31



Master dataset preview:


,player_id,year,player_name,usage_percent,avg_speed,avg_spin,avg_break_x,avg_break_z,avg_break_z_induced,avg_break,...,profile_variant,multiple_profiles,pa,k_percent,bb_percent,barrel_batted_rate,hard_hit_percent,rockies_games,rockies_innings,rockies
0,112526,2015,"Colon, Bartolo",29.1,90.9,2255.0,-6.9,-18.2,15.5,17.2,...,1,False,815,16.7,2.9,6.1,34.7,NaN,NaN,N
1,115629,2015,"Hawkins, LaTroy",73.8,93.7,2052.0,-12.2,-18.6,13.0,18.2,...,1,False,162,21.0,4.3,5.7,38.8,24.0,22.1,Y
2,150116,2015,"Wolf, Randy",15.7,89.3,2103.0,6.7,-17.8,16.9,18.4,...,1,False,161,17.4,9.3,5.8,41.0,NaN,NaN,N
3,150359,2015,"Burnett, A.J.",11.7,91.7,2082.0,-6.2,-21.0,12.0,13.8,...,1,False,699,20.5,7.0,5.6,38.7,NaN,NaN,N
4,217096,2015,"Zito, Barry",30.0,84.0,2084.0,3.8,-20.2,19.1,19.6,...,1,False,37,5.4,16.2,17.9,31.0,NaN,NaN,N
5,276514,2015,"Gregg, Kevin",62.0,91.5,2236.0,-11.7,-17.1,16.1,20.1,...,1,False,51,27.5,9.8,12.9,53.1,NaN,NaN,N
6,276542,2015,"Benoit, Joaquín",38.0,94.8,2280.0,-9.4,-15.2,15.8,18.6,...,1,False,254,24.8,9.1,5.8,24.7,NaN,NaN,N
7,279824,2015,"Buehrle, Mark",26.4,84.5,2076.0,2.0,-26.8,12.1,12.7,...,1,False,827,11.0,4.0,5.4,33.0,NaN,NaN,N
8,285064,2015,"Vogelsong, Ryan",41.2,91.6,2071.0,-7.9,-18.2,14.9,17.2,...,1,False,598,18.1,9.7,8.8,36.1,NaN,NaN,N
9,346798,2015,"Lohse, Kyle",0.9,89.8,2294.0,-3.5,-16.0,18.4,18.8,...,1,False,665,16.2,6.5,8.6,32.0,NaN,NaN,N



Saved to: /content/drive/MyDrive/Coors Project/coors_master_pitch_types_long.csv


### Step 4 Final Data Quality Check


In [4]:
# ============================================================
# STEP 4: MASTER DATASET QUALITY AUDIT
# ============================================================

import pandas as pd

# Load our saved datasets
master = pd.read_csv(
    DATA_PATH / "coors_master_pitch_types_long.csv"
)

rockies = pd.read_csv(
    DATA_PATH / "rockies_pitchers_2015_2026.csv"
)

keys = ["player_id", "year"]

# ------------------------------------------------------------
# 1. Identify Rockies pitcher-seasons missing from Savant
# ------------------------------------------------------------

savant_keys = master[keys].drop_duplicates()

missing_rockies = rockies.merge(
    savant_keys,
    on=keys,
    how="left",
    indicator=True
)

missing_rockies = missing_rockies[
    missing_rockies["_merge"] == "left_only"
].drop(columns="_merge")

print("=" * 55)
print("MISSING ROCKIES PITCHER-SEASONS")
print("=" * 55)

print("Missing records:", len(missing_rockies))

display(
    missing_rockies[
        ["player_name", "year", "games", "innings_pitched"]
    ].sort_values(["year", "player_name"])
)

# ------------------------------------------------------------
# 2. Examine multiple pitch profiles
# ------------------------------------------------------------

duplicate_profiles = (
    master
    .groupby(["player_id", "year", "pitch_type"])
    .agg(
        player_name=("player_name", "first"),
        profiles=("pitch_type", "size")
    )
    .reset_index()
)

duplicate_profiles = duplicate_profiles[
    duplicate_profiles["profiles"] > 1
]

print("\n" + "=" * 55)
print("MULTIPLE PITCH PROFILES")
print("=" * 55)

print("Duplicate profile groups:", len(duplicate_profiles))

display(
    duplicate_profiles.sort_values(
        ["player_name", "year", "pitch_type"]
    )
)

# ------------------------------------------------------------
# 3. Confirm pitch-type distribution
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("PITCH TYPE COVERAGE")
print("=" * 55)

pitch_coverage = (
    master
    .groupby("pitch_type")
    .agg(
        records=("player_id", "size"),
        unique_pitchers=("player_id", "nunique"),
        rockies_records=(
            "rockies",
            lambda x: (x == "Y").sum()
        )
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

display(pitch_coverage)

MISSING ROCKIES PITCHER-SEASONS
Missing records: 13


,player_name,year,games,innings_pitched
135,Drew Butera,2020,1,1.2
165,Joshua Fuentes,2021,1,1.0
182,Brian Serven,2022,1,1.0
203,Alan Trejo,2023,2,2.0
218,Harold Castro,2023,2,2.0
231,Riley Pint,2023,1,0.1
245,Evan Justice,2024,1,0.1
259,Matt Koch,2024,2,0.1
270,Alan Trejo,2025,1,1.0
275,Austin Nola,2025,1,1.0



MULTIPLE PITCH PROFILES
Duplicate profile groups: 29


,player_id,year,pitch_type,player_name,profiles
12939,519381,2015,breaking_group,"Venditte, Pat",2
12940,519381,2015,changeup,"Venditte, Pat",2
12941,519381,2015,fastball_group,"Venditte, Pat",2
12942,519381,2015,offspeed_group,"Venditte, Pat",2
12943,519381,2015,sinker,"Venditte, Pat",2
12944,519381,2015,slider,"Venditte, Pat",2
12945,519381,2016,breaking_group,"Venditte, Pat",2
12946,519381,2016,changeup,"Venditte, Pat",2
12947,519381,2016,fastball_group,"Venditte, Pat",2
12948,519381,2016,offspeed_group,"Venditte, Pat",2



PITCH TYPE COVERAGE


,pitch_type,records,unique_pitchers,rockies_records
4,fastball_group,9264,2465,320
0,breaking_group,9227,2525,316
6,four_seam,8802,2407,302
8,offspeed_group,7824,2269,278
1,changeup,6965,2096,257
11,slider,6804,2069,237
10,sinker,6619,1967,228
2,curveball,5454,1691,193
3,cutter,3110,1041,112
14,sweeper,1775,633,55


### Step 6 — Identify actual transitions between organizations


In [6]:
# ============================================================
# STEP 6: MATCH PITCH PROFILES ACROSS TEAM TRANSITIONS
# ============================================================

import pandas as pd
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)

master = pd.read_csv(
    DATA_PATH / "coors_master_pitch_types_long.csv"
)

MIN_PA = 100
MIN_USAGE = 5
MAX_YEAR_GAP = 2

# Keep actual pitch types, not broad groups
df = master[
    master["is_group"].astype(str).str.lower() == "false"
].copy()

# Exclude switch-pitching profiles pending handedness resolution
df = df[df["player_id"] != 519381].copy()

# Convert numeric measurements
numeric_cols = [
    "pa",
    "usage_percent",
    "avg_break_z_induced",
    "avg_break_x",
    "avg_speed",
    "avg_spin"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Apply exploratory workload and pitch-usage filters
df = df[
    (df["pa"] >= MIN_PA)
    & (df["usage_percent"] >= MIN_USAGE)
].copy()

# Require induced vertical break
df = df.dropna(subset=["avg_break_z_induced"])

# Each eligible pitcher-season-pitch must be unique
assert not df.duplicated(
    ["player_id", "year", "pitch_type"]
).any(), "Duplicate profiles need review."

# ------------------------------------------------------------
# Identify neighboring observations where Rockies status changes
# ------------------------------------------------------------

pairs = []

for (player_id, pitch_type), group in df.groupby(
    ["player_id", "pitch_type"]
):

    group = group.sort_values("year").reset_index(drop=True)

    for i in range(1, len(group)):

        previous = group.iloc[i - 1]
        current = group.iloc[i]

        year_gap = int(current["year"] - previous["year"])

        # Require a change between Rockies and other teams
        if previous["rockies"] == current["rockies"]:
            continue

        if year_gap > MAX_YEAR_GAP:
            continue

        if previous["rockies"] == "Y":
            colorado = previous
            other = current
            direction = "Leaving Colorado"
        else:
            colorado = current
            other = previous
            direction = "Joining Colorado"

        pairs.append({
            "player_id": player_id,
            "player_name": colorado["player_name"],
            "pitch_type": pitch_type,
            "transition": direction,
            "rockies_year": int(colorado["year"]),
            "other_year": int(other["year"]),
            "year_gap": year_gap,
            "rockies_ivb": colorado["avg_break_z_induced"],
            "other_ivb": other["avg_break_z_induced"],
            "ivb_difference": (
                colorado["avg_break_z_induced"]
                - other["avg_break_z_induced"]
            ),
            "rockies_velocity": colorado["avg_speed"],
            "other_velocity": other["avg_speed"],
            "rockies_spin": colorado["avg_spin"],
            "other_spin": other["avg_spin"],
            "rockies_usage": colorado["usage_percent"],
            "other_usage": other["usage_percent"],
            "rockies_pa": colorado["pa"],
            "other_pa": other["pa"],
            "crosses_2020_tracking_change": (
                min(int(colorado["year"]), int(other["year"])) < 2020
                <= max(int(colorado["year"]), int(other["year"]))
            )
        })

transitions_df = pd.DataFrame(pairs)

if transitions_df.empty:
    raise ValueError(
        "No eligible transitions found. Review sample thresholds."
    )

# Save the analysis-ready transition dataset
transitions_df.to_csv(
    DATA_PATH / "coors_pitch_transitions.csv",
    index=False
)

# ------------------------------------------------------------
# Summarize eligibility
# ------------------------------------------------------------

print("=" * 55)
print("MATCHED PITCH TRANSITION ANALYSIS")
print("=" * 55)

print("Total comparisons:", len(transitions_df))

print(
    "Unique pitchers:",
    transitions_df["player_id"].nunique()
)

print("\nComparison direction:")
print(transitions_df["transition"].value_counts())

print("\nComparable pitchers by pitch type:")

pitch_counts = (
    transitions_df
    .groupby("pitch_type")
    .agg(
        comparisons=("player_id", "size"),
        unique_pitchers=("player_id", "nunique")
    )
    .reset_index()
    .sort_values("unique_pitchers", ascending=False)
)

display(pitch_counts)

print("\nSample transitions:")
display(
    transitions_df[
        [
            "player_name",
            "pitch_type",
            "transition",
            "other_year",
            "rockies_year",
            "other_ivb",
            "rockies_ivb",
            "ivb_difference"
        ]
    ].head(20).round(2)
)

print("\nSaved: coors_pitch_transitions.csv")

MATCHED PITCH TRANSITION ANALYSIS
Total comparisons: 233
Unique pitchers: 55

Comparison direction:
transition
Leaving Colorado    127
Joining Colorado    106
Name: count, dtype: int64

Comparable pitchers by pitch type:


,pitch_type,comparisons,unique_pitchers
3,four_seam,59,45
5,slider,39,30
4,sinker,32,27
1,curveball,31,27
0,changeup,29,23
2,cutter,24,17
8,sweeper,12,11
7,splitter,5,4
6,slurve,2,2



Sample transitions:


,player_name,pitch_type,transition,other_year,rockies_year,other_ivb,rockies_ivb,ivb_difference
0,"De La Rosa, Jorge",curveball,Leaving Colorado,2017,2016,-11.2,-10.6,0.6
1,"De La Rosa, Jorge",cutter,Leaving Colorado,2017,2016,8.7,7.5,-1.2
2,"De La Rosa, Jorge",four_seam,Leaving Colorado,2017,2016,14.0,11.9,-2.1
3,"De La Rosa, Jorge",splitter,Leaving Colorado,2017,2016,3.0,2.4,-0.6
4,"Qualls, Chad",sinker,Joining Colorado,2015,2016,4.5,4.5,0.0
5,"Qualls, Chad",slider,Joining Colorado,2015,2016,4.6,3.0,-1.6
6,"Motte, Jason",cutter,Joining Colorado,2015,2016,7.6,7.3,-0.3
7,"Motte, Jason",cutter,Leaving Colorado,2017,2016,5.8,7.3,1.5
8,"Motte, Jason",four_seam,Joining Colorado,2015,2016,16.7,15.7,-1.0
9,"Motte, Jason",four_seam,Leaving Colorado,2017,2016,19.4,15.7,-3.7



Saved: coors_pitch_transitions.csv


### Step 7 — Calculate first pitch movement findings


In [7]:
# ============================================================
# STEP 7: PRELIMINARY PITCH MOVEMENT RESULTS
# ============================================================

import pandas as pd
import numpy as np

# Load transition comparisons
df = pd.read_csv(
    DATA_PATH / "coors_pitch_transitions.csv"
)

# Exclude transitions spanning the tracking-system change
tracking_change = (
    df["crosses_2020_tracking_change"]
    .astype(str)
    .str.lower()
    .eq("true")
)

# Use adjacent seasons only
clean = df[
    (df["year_gap"] == 1)
    & (~tracking_change)
].copy()

print("Original comparisons:", len(df))
print("Comparisons after filtering:", len(clean))
print("Unique pitchers:", clean["player_id"].nunique())

# Calculate Rockies minus non-Rockies differences
clean["velocity_difference"] = (
    clean["rockies_velocity"] - clean["other_velocity"]
)

clean["spin_difference"] = (
    clean["rockies_spin"] - clean["other_spin"]
)

# One observation per pitcher and pitch type
pitcher_results = (
    clean.groupby(["player_id", "pitch_type"])
    .agg(
        player_name=("player_name", "first"),
        comparisons=("ivb_difference", "size"),
        ivb_difference=("ivb_difference", "mean"),
        velocity_difference=("velocity_difference", "mean"),
        spin_difference=("spin_difference", "mean")
    )
    .reset_index()
)

# Bootstrap confidence intervals across pitchers
rng = np.random.default_rng(42)

def bootstrap_ci(values, n_boot=5000):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 5:
        return np.nan, np.nan

    samples = rng.choice(
        values,
        size=(n_boot, len(values)),
        replace=True
    )

    means = samples.mean(axis=1)

    return np.percentile(means, [2.5, 97.5])

results = []

for pitch_type, group in pitcher_results.groupby("pitch_type"):

    values = group["ivb_difference"].dropna().to_numpy()

    ci_low, ci_high = bootstrap_ci(values)

    results.append({
        "pitch_type": pitch_type,
        "pitchers": len(values),
        "mean_ivb_change": values.mean() if len(values) else np.nan,
        "median_ivb_change": np.median(values) if len(values) else np.nan,
        "ci_lower": ci_low,
        "ci_upper": ci_high,
        "mean_velocity_change": group["velocity_difference"].mean(),
        "mean_spin_change": group["spin_difference"].mean()
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("pitchers", ascending=False)
)

print("\nPITCH MOVEMENT DIFFERENCES")
print("All differences = Rockies season minus non-Rockies season")
print("IVB: inches | Velocity: mph | Spin: rpm")

display(results_df.round(2))

# Save results
pitcher_results.to_csv(
    DATA_PATH / "coors_pitcher_level_results.csv",
    index=False
)

results_df.to_csv(
    DATA_PATH / "coors_preliminary_pitch_results.csv",
    index=False
)

Original comparisons: 233
Comparisons after filtering: 189
Unique pitchers: 49

PITCH MOVEMENT DIFFERENCES
All differences = Rockies season minus non-Rockies season
IVB: inches | Velocity: mph | Spin: rpm


,pitch_type,pitchers,mean_ivb_change,median_ivb_change,ci_lower,ci_upper,mean_velocity_change,mean_spin_change
3,four_seam,40,-1.42,-1.60,-1.80,-1.03,-0.15,14.15
5,slider,25,-0.65,-0.30,-1.50,0.18,-0.16,8.78
1,curveball,24,0.78,0.65,-0.04,1.65,-0.15,38.83
4,sinker,23,-1.02,-1.10,-1.75,-0.35,0.03,4.39
0,changeup,19,-0.77,-0.70,-1.31,-0.25,-0.19,1.50
2,cutter,16,-1.41,-1.22,-2.22,-0.65,-0.14,-3.72
8,sweeper,9,0.96,0.80,0.12,1.89,0.36,8.11
7,splitter,4,-1.30,-0.65,NaN,NaN,-0.84,-25.25
6,slurve,1,1.30,1.30,NaN,NaN,1.50,66.00


### Step 8 — Test consistency across transition directions


In [8]:
# ============================================================
# STEP 8: CHECK CONSISTENCY OF PITCH MOVEMENT CHANGES
# ============================================================

import pandas as pd

df = pd.read_csv(
    DATA_PATH / "coors_pitch_transitions.csv"
)

# Reproduce Step 7 filters
tracking_change = (
    df["crosses_2020_tracking_change"]
    .astype(str)
    .str.lower()
    .eq("true")
)

df = df[
    (df["year_gap"] == 1)
    & (~tracking_change)
].copy()

primary_pitches = [
    "four_seam",
    "sinker",
    "slider",
    "curveball",
    "changeup",
    "cutter"
]

df = df[df["pitch_type"].isin(primary_pitches)]

# Average repeated comparisons for the same pitcher,
# pitch type, and transition direction.
pitcher_direction = (
    df.groupby(
        ["player_id", "pitch_type", "transition"],
        as_index=False
    )
    .agg(
        ivb_difference=("ivb_difference", "mean")
    )
)

# Compare joining and leaving Colorado
direction_results = (
    pitcher_direction
    .groupby(["pitch_type", "transition"])
    .agg(
        pitchers=("player_id", "nunique"),
        mean_ivb_change=("ivb_difference", "mean"),
        median_ivb_change=("ivb_difference", "median"),
        pct_negative=(
            "ivb_difference",
            lambda x: 100 * (x < 0).mean()
        )
    )
    .reset_index()
)

print("IVB DIFFERENCES BY TRANSITION DIRECTION")
print("All differences are Rockies minus non-Rockies.")
print()

display(direction_results.round(2))

# Overall consistency: one value per pitcher and pitch
pitcher_average = (
    df.groupby(["player_id", "pitch_type"])
    .agg(ivb_difference=("ivb_difference", "mean"))
    .reset_index()
)

consistency = (
    pitcher_average
    .groupby("pitch_type")
    .agg(
        pitchers=("player_id", "nunique"),
        mean_change=("ivb_difference", "mean"),
        median_change=("ivb_difference", "median"),
        pct_negative=(
            "ivb_difference",
            lambda x: 100 * (x < 0).mean()
        ),
        pct_positive=(
            "ivb_difference",
            lambda x: 100 * (x > 0).mean()
        )
    )
    .reset_index()
)

print("\nOVERALL PITCHER-LEVEL CONSISTENCY")

display(
    consistency.round(2)
    .sort_values("pitchers", ascending=False)
)

IVB DIFFERENCES BY TRANSITION DIRECTION
All differences are Rockies minus non-Rockies.



,pitch_type,transition,pitchers,mean_ivb_change,median_ivb_change,pct_negative
0,changeup,Joining Colorado,10,-0.66,-0.50,80.00
1,changeup,Leaving Colorado,14,-0.80,-1.30,71.43
2,curveball,Joining Colorado,10,0.66,-0.40,50.00
3,curveball,Leaving Colorado,16,0.75,0.65,31.25
4,cutter,Joining Colorado,13,-1.78,-1.10,92.31
5,cutter,Leaving Colorado,8,-0.82,-1.30,75.00
6,four_seam,Joining Colorado,20,-1.53,-1.40,95.00
7,four_seam,Leaving Colorado,27,-1.36,-1.70,81.48
8,sinker,Joining Colorado,15,-1.15,-0.70,73.33
9,sinker,Leaving Colorado,13,-0.57,-0.70,53.85



OVERALL PITCHER-LEVEL CONSISTENCY


,pitch_type,pitchers,mean_change,median_change,pct_negative,pct_positive
3,four_seam,40,-1.42,-1.60,87.50,10.00
5,slider,25,-0.65,-0.30,60.00,40.00
1,curveball,24,0.78,0.65,33.33,66.67
4,sinker,23,-1.02,-1.10,69.57,26.09
0,changeup,19,-0.77,-0.70,73.68,26.32
2,cutter,16,-1.41,-1.22,81.25,18.75


### Step 10 — Calculate Colorado innings share for every transition pitcher


In [10]:
# ============================================================
# STEP 10: CALCULATE ROCKIES INNINGS SHARE
# ============================================================

import pandas as pd
import requests
import json
import time
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)
CACHE_PATH = DATA_PATH / "MLB_GameLogs"
CACHE_PATH.mkdir(parents=True, exist_ok=True)

# Load transition data
transitions = pd.read_csv(
    DATA_PATH / "coors_pitch_transitions.csv"
)

# Identify unique Rockies player-seasons
rockies_seasons = (
    transitions[["player_id", "rockies_year"]]
    .drop_duplicates()
    .rename(columns={"rockies_year": "year"})
    .sort_values(["year", "player_id"])
)

def innings_to_outs(ip):
    if pd.isna(ip):
        return 0

    parts = str(ip).split(".")
    innings = int(parts[0])
    extra = int(parts[1]) if len(parts) > 1 else 0

    if extra not in (0, 1, 2):
        raise ValueError(f"Invalid innings: {ip}")

    return innings * 3 + extra


records = []
failures = []

for row in rockies_seasons.itertuples(index=False):

    pid = int(row.player_id)
    year = int(row.year)

    cache_file = CACHE_PATH / f"{pid}_{year}.json"

    try:
        if cache_file.exists():
            with open(cache_file) as f:
                data = json.load(f)
        else:
            url = (
                f"https://statsapi.mlb.com/api/v1/"
                f"people/{pid}/stats"
            )

            response = requests.get(
                url,
                params={
                    "stats": "gameLog",
                    "group": "pitching",
                    "season": year,
                    "gameType": "R"
                },
                timeout=45
            )

            response.raise_for_status()
            data = response.json()

            with open(cache_file, "w") as f:
                json.dump(data, f)

            time.sleep(0.15)

        splits = [
            split
            for block in data.get("stats", [])
            for split in block.get("splits", [])
        ]

        if not splits:
            raise ValueError("No game logs returned")

        team_outs = {}

        for split in splits:
            team = split.get("team", {})
            team_id = team.get("id")

            if team_id is None:
                raise ValueError("Missing team ID")

            outs = innings_to_outs(
                split.get("stat", {}).get("inningsPitched")
            )

            team_outs[team_id] = (
                team_outs.get(team_id, 0) + outs
            )

        total_outs = sum(team_outs.values())
        colorado_outs = team_outs.get(115, 0)

        if total_outs == 0 or colorado_outs == 0:
            raise ValueError("Missing positive Rockies innings")

        records.append({
            "player_id": pid,
            "rockies_year": year,
            "total_innings": total_outs / 3,
            "colorado_innings": colorado_outs / 3,
            "colorado_share": colorado_outs / total_outs,
            "teams_played_for": len(team_outs)
        })

    except Exception as e:
        failures.append({
            "player_id": pid,
            "year": year,
            "error": str(e)
        })

# Create results
innings_share = pd.DataFrame(records)

if not innings_share.empty:

    innings_share["qualifies_80pct"] = (
        innings_share["colorado_share"] >= 0.80
    )

    innings_share.to_csv(
        DATA_PATH / "rockies_innings_share.csv",
        index=False
    )

print("=" * 55)
print("ROCKIES INNINGS SHARE ANALYSIS")
print("=" * 55)

print("Seasons checked:", len(rockies_seasons))
print("Successful:", len(records))
print("Failed:", len(failures))

if not innings_share.empty:

    print("\nSeasons with at least 80% Colorado innings:")
    print(innings_share["qualifies_80pct"].sum())

    print("\nMixed-team seasons below 80%:")
    display(
        innings_share[
            ~innings_share["qualifies_80pct"]
        ].sort_values("colorado_share").round(3)
    )

    print("\nSample of qualified seasons:")
    display(
        innings_share[
            innings_share["qualifies_80pct"]
        ].head(10).round(3)
    )

if failures:
    print("\nUnverified seasons:")
    display(pd.DataFrame(failures))
    print(
        "Do not include these seasons in the primary analysis "
        "until their team shares are verified."
    )

ROCKIES INNINGS SHARE ANALYSIS
Seasons checked: 63
Successful: 63
Failed: 0

Seasons with at least 80% Colorado innings:
48

Mixed-team seasons below 80%:


,player_id,rockies_year,total_innings,colorado_innings,colorado_share,teams_played_for,qualifies_80pct
45,677865,2023,28.000,3.667,0.131,2,False
17,493200,2018,68.333,21.333,0.312,2,False
12,450212,2017,62.333,22.000,0.353,2,False
41,570632,2023,44.667,18.333,0.410,2,False
27,571710,2021,51.000,29.667,0.582,2,False
44,623167,2023,102.333,60.333,0.590,2,False
38,659275,2022,32.333,20.000,0.619,2,False
42,572955,2023,62.667,39.000,0.622,2,False
54,641755,2025,72.667,47.667,0.656,2,False
40,543272,2023,53.667,35.667,0.665,2,False



Sample of qualified seasons:


,player_id,rockies_year,total_innings,colorado_innings,colorado_share,teams_played_for,qualifies_80pct
0,446099,2015,55.667,55.667,1.0,1,True
1,543184,2015,58.333,58.333,1.0,1,True
2,548357,2015,68.333,68.333,1.0,1,True
3,592454,2015,33.333,33.333,1.0,1,True
4,407822,2016,134.000,134.000,1.0,1,True
5,430589,2016,32.667,32.667,1.0,1,True
6,435400,2016,23.667,23.667,1.0,1,True
7,459429,2016,45.667,45.667,1.0,1,True
8,502522,2016,42.667,42.667,1.0,1,True
9,548357,2016,24.667,24.667,1.0,1,True


### Step 11 — Recalculate pitch movement with our verified sample


In [11]:
# ============================================================
# STEP 11: VERIFY OUR PITCH MOVEMENT FINDINGS
# Apply 80% Rockies innings threshold
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)

# Load our existing datasets
transitions = pd.read_csv(
    DATA_PATH / "coors_pitch_transitions.csv"
)

innings_share = pd.read_csv(
    DATA_PATH / "rockies_innings_share.csv"
)

# ------------------------------------------------------------
# 1. Reproduce our previous analysis filters
# ------------------------------------------------------------

tracking_change = (
    transitions["crosses_2020_tracking_change"]
    .astype(str)
    .str.lower()
    .eq("true")
)

primary_pitches = [
    "four_seam",
    "sinker",
    "slider",
    "curveball",
    "changeup",
    "cutter",
    "sweeper"
]

baseline = transitions[
    (transitions["year_gap"] == 1)
    & (~tracking_change)
    & (transitions["pitch_type"].isin(primary_pitches))
].copy()

# ------------------------------------------------------------
# 2. Attach verified Colorado innings shares
# ------------------------------------------------------------

share_lookup = innings_share[
    ["player_id", "rockies_year", "colorado_share"]
].copy()

assert not share_lookup.duplicated(
    ["player_id", "rockies_year"]
).any(), "Duplicate innings-share records found"

verified = baseline.merge(
    share_lookup,
    on=["player_id", "rockies_year"],
    how="left",
    validate="many_to_one"
)

# Ensure every transition has a verified share
if verified["colorado_share"].isna().any():
    raise ValueError(
        "Some Rockies seasons are missing verified innings shares."
    )

# Apply 80% threshold
qualified = verified[
    verified["colorado_share"] >= 0.80
].copy()

# Save filtered transitions
qualified.to_csv(
    DATA_PATH / "coors_transitions_80pct.csv",
    index=False
)

# ------------------------------------------------------------
# 3. One average difference per pitcher and pitch type
# ------------------------------------------------------------

def pitcher_level(df):

    return (
        df.groupby(["player_id", "pitch_type"])
        .agg(
            ivb_difference=("ivb_difference", "mean"),
            comparisons=("ivb_difference", "size")
        )
        .reset_index()
    )

baseline_pitchers = pitcher_level(baseline)
qualified_pitchers = pitcher_level(qualified)

# ------------------------------------------------------------
# 4. Bootstrap confidence intervals across pitchers
# ------------------------------------------------------------

rng = np.random.default_rng(42)

def bootstrap_ci(values, iterations=5000):

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 5:
        return np.nan, np.nan

    samples = rng.choice(
        values,
        size=(iterations, len(values)),
        replace=True
    )

    sample_means = samples.mean(axis=1)

    return np.percentile(
        sample_means, [2.5, 97.5]
    )

# ------------------------------------------------------------
# 5. Summarize each pitch type
# ------------------------------------------------------------

results = []

for pitch_type in primary_pitches:

    old = baseline_pitchers[
        baseline_pitchers["pitch_type"] == pitch_type
    ]

    new = qualified_pitchers[
        qualified_pitchers["pitch_type"] == pitch_type
    ]

    values = new["ivb_difference"].dropna().to_numpy()

    if len(values) == 0:
        continue

    ci_lower, ci_upper = bootstrap_ci(values)

    results.append({
        "pitch_type": pitch_type,
        "original_pitchers": len(old),
        "qualified_pitchers": len(values),
        "original_mean_ivb": old["ivb_difference"].mean(),
        "qualified_mean_ivb": values.mean(),
        "qualified_median_ivb": np.median(values),
        "pct_lower_ivb": 100 * (values < 0).mean(),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper
    })

results_80 = pd.DataFrame(results)

results_80 = results_80.sort_values(
    "qualified_pitchers",
    ascending=False
)

# ------------------------------------------------------------
# 6. Print results
# ------------------------------------------------------------

print("=" * 60)
print("VERIFIED COORS PITCH MOVEMENT ANALYSIS")
print("=" * 60)

print("Original filtered comparisons:", len(baseline))
print("Qualified comparisons:", len(qualified))

print(
    "Unique qualified pitchers:",
    qualified["player_id"].nunique()
)

print("\nORIGINAL VS VERIFIED PITCH MOVEMENT")

display(results_80.round(2))

# Save results
results_80.to_csv(
    DATA_PATH / "coors_ivb_results_verified.csv",
    index=False
)

qualified_pitchers.to_csv(
    DATA_PATH / "coors_pitcher_results_verified.csv",
    index=False
)

print("\nResults saved successfully.")

VERIFIED COORS PITCH MOVEMENT ANALYSIS
Original filtered comparisons: 183
Qualified comparisons: 132
Unique qualified pitchers: 35

ORIGINAL VS VERIFIED PITCH MOVEMENT


,pitch_type,original_pitchers,qualified_pitchers,original_mean_ivb,qualified_mean_ivb,qualified_median_ivb,pct_lower_ivb,ci_lower,ci_upper
0,four_seam,40,28,-1.42,-1.59,-1.90,85.71,-2.07,-1.09
3,curveball,24,21,0.78,0.57,0.60,38.10,-0.24,1.46
1,sinker,23,19,-1.02,-1.24,-1.15,78.95,-1.99,-0.52
2,slider,25,16,-0.65,-0.57,-0.20,56.25,-1.66,0.43
4,changeup,19,14,-0.77,-0.95,-1.12,85.71,-1.44,-0.41
5,cutter,16,12,-1.41,-1.88,-1.45,91.67,-2.74,-1.06
6,sweeper,9,7,0.96,0.56,0.30,28.57,-0.19,1.37



Results saved successfully.


### Step 12 — What happens if we use 100% Rockies seasons?


In [12]:
# ============================================================
# STEP 12: TEST SENSITIVITY TO ROCKIES INNINGS THRESHOLD
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)

# Load our verified transition comparisons
df = pd.read_csv(
    DATA_PATH / "coors_transitions_80pct.csv"
)

df["colorado_share"] = pd.to_numeric(
    df["colorado_share"], errors="coerce"
)

df["ivb_difference"] = pd.to_numeric(
    df["ivb_difference"], errors="coerce"
)

# Compare our existing 80% sample against
# seasons with 100% of innings for Colorado
sample_80 = df[df["colorado_share"] >= 0.80].copy()

sample_100 = df[
    np.isclose(df["colorado_share"], 1.0, atol=1e-9)
].copy()


def summarize_sample(data, sample_name):

    # Average repeated transitions for each pitcher and pitch
    pitcher_level = (
        data.groupby(["player_id", "pitch_type"], as_index=False)
        .agg(ivb_difference=("ivb_difference", "mean"))
    )

    summary = (
        pitcher_level.groupby("pitch_type")
        .agg(
            pitchers=("player_id", "nunique"),
            mean_ivb=("ivb_difference", "mean"),
            median_ivb=("ivb_difference", "median"),
            pct_lower=(
                "ivb_difference",
                lambda x: 100 * (x < 0).mean()
            )
        )
        .reset_index()
    )

    summary["sample"] = sample_name

    return summary


results_80 = summarize_sample(sample_80, "80% Rockies")
results_100 = summarize_sample(sample_100, "100% Rockies")

# Combine results
sensitivity_results = pd.concat(
    [results_80, results_100],
    ignore_index=True
)

sensitivity_results = sensitivity_results[
    [
        "pitch_type",
        "sample",
        "pitchers",
        "mean_ivb",
        "median_ivb",
        "pct_lower"
    ]
].sort_values(["pitch_type", "sample"])

print("=" * 60)
print("ROCKIES INNINGS THRESHOLD SENSITIVITY")
print("=" * 60)

print(
    "80% sample unique pitchers:",
    sample_80["player_id"].nunique()
)

print(
    "100% sample unique pitchers:",
    sample_100["player_id"].nunique()
)

display(sensitivity_results.round(2))

# Save
sensitivity_results.to_csv(
    DATA_PATH / "coors_threshold_sensitivity.csv",
    index=False
)

ROCKIES INNINGS THRESHOLD SENSITIVITY
80% sample unique pitchers: 35
100% sample unique pitchers: 31


,pitch_type,sample,pitchers,mean_ivb,median_ivb,pct_lower
7,changeup,100% Rockies,11,-0.97,-1.30,81.82
0,changeup,80% Rockies,14,-0.95,-1.12,85.71
8,curveball,100% Rockies,19,0.46,0.60,42.11
1,curveball,80% Rockies,21,0.57,0.60,38.10
9,cutter,100% Rockies,10,-1.65,-1.30,90.00
2,cutter,80% Rockies,12,-1.88,-1.45,91.67
10,four_seam,100% Rockies,25,-1.61,-2.00,84.00
3,four_seam,80% Rockies,28,-1.59,-1.90,85.71
11,sinker,100% Rockies,15,-1.19,-1.15,80.00
4,sinker,80% Rockies,19,-1.24,-1.15,78.95


### Step 13 — Analyze horizontal pitch movement


In [13]:
# ============================================================
# STEP 13: HORIZONTAL PITCH MOVEMENT ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)

# Load verified transition comparisons
transitions = pd.read_csv(
    DATA_PATH / "coors_transitions_80pct.csv"
)

# Load pitch measurements
master = pd.read_csv(
    DATA_PATH / "coors_master_pitch_types_long.csv"
)

# ------------------------------------------------------------
# 1. Prepare horizontal movement measurements
# ------------------------------------------------------------

pitch_data = master[
    master["is_group"].astype(str).str.lower().eq("false")
    & master["player_id"].ne(519381)
][
    ["player_id", "year", "pitch_type", "avg_break_x"]
].copy()

pitch_data["avg_break_x"] = pd.to_numeric(
    pitch_data["avg_break_x"],
    errors="coerce"
)

keys = ["player_id", "year", "pitch_type"]

assert not pitch_data.duplicated(keys).any(), (
    "Duplicate pitch profiles need review."
)

# ------------------------------------------------------------
# 2. Attach Rockies and non-Rockies horizontal movement
# ------------------------------------------------------------

rockies_movement = pitch_data.rename(
    columns={
        "year": "rockies_year",
        "avg_break_x": "rockies_horizontal"
    }
)

other_movement = pitch_data.rename(
    columns={
        "year": "other_year",
        "avg_break_x": "other_horizontal"
    }
)

df = transitions.merge(
    rockies_movement,
    on=["player_id", "rockies_year", "pitch_type"],
    how="left",
    validate="many_to_one"
)

df = df.merge(
    other_movement,
    on=["player_id", "other_year", "pitch_type"],
    how="left",
    validate="many_to_one"
)

if df[["rockies_horizontal", "other_horizontal"]].isna().any().any():
    raise ValueError(
        "Missing horizontal movement for some matched transitions."
    )

# ------------------------------------------------------------
# 3. Calculate movement magnitude differences
# ------------------------------------------------------------

df["horizontal_change"] = (
    df["rockies_horizontal"].abs()
    - df["other_horizontal"].abs()
)

# ------------------------------------------------------------
# 4. Compare 80% and 100% Rockies samples
# ------------------------------------------------------------

results = []

pitch_types = [
    "four_seam", "sinker", "slider",
    "curveball", "changeup", "cutter", "sweeper"
]

for sample_name, threshold in [
    ("80% Rockies", 0.80),
    ("100% Rockies", 1.00)
]:

    if threshold == 1.00:
        subset = df[
            np.isclose(df["colorado_share"], 1.0)
        ].copy()
    else:
        subset = df[
            df["colorado_share"] >= threshold
        ].copy()

    # One average observation per pitcher and pitch type
    pitcher_level = (
        subset.groupby(["player_id", "pitch_type"])
        .agg(
            horizontal_change=("horizontal_change", "mean")
        )
        .reset_index()
    )

    for pitch_type in pitch_types:

        values = pitcher_level.loc[
            pitcher_level["pitch_type"] == pitch_type,
            "horizontal_change"
        ].dropna()

        if len(values) == 0:
            continue

        results.append({
            "pitch_type": pitch_type,
            "sample": sample_name,
            "pitchers": len(values),
            "mean_change": values.mean(),
            "median_change": values.median(),
            "pct_less_movement": 100 * (values < 0).mean()
        })

horizontal_results = pd.DataFrame(results)

print("=" * 60)
print("HORIZONTAL PITCH MOVEMENT ANALYSIS")
print("=" * 60)

print(
    "Movement change = Rockies magnitude "
    "minus non-Rockies magnitude"
)

display(
    horizontal_results
    .sort_values(["pitch_type", "sample"])
    .round(2)
)

# Save the results
horizontal_results.to_csv(
    DATA_PATH / "coors_horizontal_movement_results.csv",
    index=False
)

print("\nHorizontal movement results saved.")

HORIZONTAL PITCH MOVEMENT ANALYSIS
Movement change = Rockies magnitude minus non-Rockies magnitude


,pitch_type,sample,pitchers,mean_change,median_change,pct_less_movement
11,changeup,100% Rockies,11,-2.53,-1.80,100.00
4,changeup,80% Rockies,14,-2.28,-1.65,100.00
10,curveball,100% Rockies,19,-0.43,-0.70,63.16
3,curveball,80% Rockies,21,-0.49,-0.70,66.67
12,cutter,100% Rockies,10,1.20,1.33,30.00
5,cutter,80% Rockies,12,1.12,1.33,33.33
7,four_seam,100% Rockies,25,-0.73,-0.90,64.00
0,four_seam,80% Rockies,28,-0.72,-0.90,64.29
8,sinker,100% Rockies,15,-1.38,-1.20,80.00
1,sinker,80% Rockies,19,-1.48,-1.30,84.21



Horizontal movement results saved.


### Step 14 — Investigate changeups and cutters individually


In [14]:
# ============================================================
# STEP 14: INVESTIGATE CHANGEUP AND CUTTER MOVEMENT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)

transitions = pd.read_csv(
    DATA_PATH / "coors_transitions_80pct.csv"
)

master = pd.read_csv(
    DATA_PATH / "coors_master_pitch_types_long.csv"
)

# Focus on our two most intriguing pitch types
focus_pitches = ["changeup", "cutter"]

# Only seasons with 100% of innings for Colorado
df = transitions[
    transitions["pitch_type"].isin(focus_pitches)
    & np.isclose(transitions["colorado_share"], 1.0)
].copy()

# Retrieve original horizontal movement measurements
pitch_data = master[
    master["pitch_type"].isin(focus_pitches)
    & master["player_id"].ne(519381)
][
    ["player_id", "year", "pitch_type", "avg_break_x"]
].copy()

keys = ["player_id", "year", "pitch_type"]

assert not pitch_data.duplicated(keys).any()

rockies_data = pitch_data.rename(columns={
    "year": "rockies_year",
    "avg_break_x": "rockies_horizontal"
})

other_data = pitch_data.rename(columns={
    "year": "other_year",
    "avg_break_x": "other_horizontal"
})

df = df.merge(
    rockies_data,
    on=["player_id", "rockies_year", "pitch_type"],
    validate="many_to_one"
)

df = df.merge(
    other_data,
    on=["player_id", "other_year", "pitch_type"],
    validate="many_to_one"
)

# Calculate absolute horizontal movement differences
df["horizontal_difference"] = (
    df["rockies_horizontal"].abs()
    - df["other_horizontal"].abs()
)

# Check whether movement direction changed
df["direction_reversed"] = (
    df["rockies_horizontal"] * df["other_horizontal"] < 0
)

# Examine each individual transition
print("INDIVIDUAL PITCH TRANSITIONS")

display(
    df[[
        "player_name",
        "pitch_type",
        "transition",
        "other_year",
        "rockies_year",
        "other_horizontal",
        "rockies_horizontal",
        "horizontal_difference",
        "direction_reversed"
    ]].sort_values(
        ["pitch_type", "horizontal_difference"]
    ).round(2)
)

# Compare joining versus leaving Colorado
print("\nRESULTS BY TRANSITION DIRECTION")

direction_summary = (
    df.groupby(["pitch_type", "transition"])
    .agg(
        pitchers=("player_id", "nunique"),
        comparisons=("player_id", "size"),
        mean_change=("horizontal_difference", "mean"),
        median_change=("horizontal_difference", "median"),
        pct_less_movement=(
            "horizontal_difference",
            lambda x: 100 * (x < 0).mean()
        )
    )
    .reset_index()
)

display(direction_summary.round(2))

print("\nMOVEMENT DIRECTION REVERSALS")
print(df.groupby("pitch_type")["direction_reversed"].sum())

# Save detailed records
df.to_csv(
    DATA_PATH / "coors_changeup_cutter_detailed.csv",
    index=False
)

INDIVIDUAL PITCH TRANSITIONS


,player_name,pitch_type,transition,other_year,rockies_year,other_horizontal,rockies_horizontal,horizontal_difference,direction_reversed
10,"Butler, Eddie",changeup,Leaving Colorado,2017,2016,-16.5,-9.4,-7.1,False
9,"Bergman, Christian",changeup,Leaving Colorado,2017,2016,-17.9,-12.0,-5.9,False
20,"Almonte, Yency",changeup,Leaving Colorado,2022,2021,-16.8,-13.8,-3.0,False
21,"Bernardino, Brennan",changeup,Joining Colorado,2025,2026,15.2,12.3,-2.9,False
11,"Gray, Jon",changeup,Leaving Colorado,2022,2021,-14.2,-11.7,-2.5,False
23,"Molina, Anthony",changeup,Leaving Colorado,2026,2025,-15.4,-13.6,-1.8,False
16,"Suter, Brent",changeup,Joining Colorado,2022,2023,10.0,8.5,-1.5,False
5,"Quintana, Jose",changeup,Joining Colorado,2025,2026,14.7,13.3,-1.4,False
14,"Estévez, Carlos",changeup,Leaving Colorado,2023,2022,-11.6,-10.6,-1.0,False
13,"Gomber, Austin",changeup,Joining Colorado,2020,2021,10.6,9.9,-0.7,False



RESULTS BY TRANSITION DIRECTION


,pitch_type,transition,pitchers,comparisons,mean_change,median_change,pct_less_movement
0,changeup,Joining Colorado,4,4,-1.62,-1.45,100.00
1,changeup,Leaving Colorado,8,8,-2.79,-2.15,100.00
2,cutter,Joining Colorado,7,7,0.47,0.40,42.86
3,cutter,Leaving Colorado,5,5,2.14,1.70,0.00



MOVEMENT DIRECTION REVERSALS
pitch_type
changeup    0
cutter      1
Name: direction_reversed, dtype: int64


### Step 15 — Test how robust the changeup finding is


In [15]:
# ============================================================
# STEP 15: CHANGEUP HORIZONTAL MOVEMENT ROBUSTNESS
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import binomtest
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR)

# Load detailed changeup and cutter transitions
detail = pd.read_csv(
    DATA_PATH / "coors_changeup_cutter_detailed.csv"
)

# Focus exclusively on changeups
changeups = detail[
    detail["pitch_type"] == "changeup"
].copy()

# Verify that no pitch changed horizontal direction
print("Direction reversals:",
      changeups["direction_reversed"].sum())

# One observation per unique pitcher
# Average both transitions if a pitcher appears twice
pitcher_level = (
    changeups
    .groupby(["player_id", "player_name"], as_index=False)
    .agg(
        movement_change=("horizontal_difference", "mean"),
        transitions=("horizontal_difference", "size")
    )
)

# Sort by movement difference
pitcher_level = pitcher_level.sort_values(
    "movement_change"
)

display(pitcher_level.round(2))

values = pitcher_level["movement_change"].to_numpy()

print("\nCHANGEUP ROBUSTNESS")
print("=" * 50)

print("Unique pitchers:", len(values))
print("Average change:", round(values.mean(), 2))
print("Median change:", round(np.median(values), 2))

print(
    "Pitchers with reduced movement:",
    int((values < 0).sum()),
    "of",
    len(values)
)

# Remove the two largest negative observations
trimmed = pitcher_level.iloc[2:]

print("\nWITHOUT TWO LARGEST REDUCTIONS")
print("Pitchers remaining:", len(trimmed))

print(
    "Average movement change:",
    round(trimmed["movement_change"].mean(), 2)
)

print(
    "Pitchers with reduced movement:",
    int((trimmed["movement_change"] < 0).sum())
)

# Bootstrap confidence interval at pitcher level
rng = np.random.default_rng(42)

bootstrap_samples = rng.choice(
    values,
    size=(10000, len(values)),
    replace=True
)

bootstrap_means = bootstrap_samples.mean(axis=1)

lower, upper = np.percentile(
    bootstrap_means, [2.5, 97.5]
)

print("\n95% BOOTSTRAP CONFIDENCE INTERVAL")
print(round(lower, 2), "to", round(upper, 2))

# Exploratory one-sided sign test
sign_test = binomtest(
    k=int((values < 0).sum()),
    n=len(values),
    p=0.5,
    alternative="greater"
)

print("\nEXPLORATORY SIGN TEST")
print("p-value:", round(sign_test.pvalue, 6))

# Save
pitcher_level.to_csv(
    DATA_PATH / "coors_changeup_robustness.csv",
    index=False
)

Direction reversals: 0


,player_id,player_name,movement_change,transitions
2,572750,"Butler, Eddie",-7.10,1
1,548357,"Bergman, Christian",-5.90,1
8,622075,"Almonte, Yency",-3.00,1
9,657514,"Bernardino, Brennan",-2.90,1
3,592351,"Gray, Jon",-2.50,1
10,683627,"Molina, Anthony",-1.80,1
0,500779,"Quintana, Jose",-1.40,1
6,608032,"Estévez, Carlos",-1.00,1
7,608718,"Suter, Brent",-0.95,2
5,596295,"Gomber, Austin",-0.70,1



CHANGEUP ROBUSTNESS
Unique pitchers: 11
Average change: -2.53
Median change: -1.8
Pitchers with reduced movement: 11 of 11

WITHOUT TWO LARGEST REDUCTIONS
Pitchers remaining: 9
Average movement change: -1.65
Pitchers with reduced movement: 9

95% BOOTSTRAP CONFIDENCE INTERVAL
-3.85 to -1.44

EXPLORATORY SIGN TEST
p-value: 0.000488
